# Universidad de la Salle
## Maestría en Inteligencia Artificial
### Aplicaciones II - Grupo 01
---
#### Realizado por:
    Ing. Camilo Andrés Castañeda Galindo

## Actividad 5: Descifrando soluciones prácticas con Transformers

### Introducción

El **Procesamiento de Lenguaje Natural (NLP)** es el área de la Inteligencia Artificial que permite a los computadores interpretar, generar y analizar lenguaje humano. Los **Transformers** son una arquitectura de red neuronal basada en mecanismos de atención que, a diferencia de los modelos recurrentes, procesan el texto completo en paralelo y capturan relaciones de largo alcance entre palabras.

El objetivo de este notebook es evaluar de forma práctica y crítica **qué tan efectivos son varios modelos Transformer preentrenados** al aplicarse a un escenario basado en la realidad de soporte técnico, sin entrenar ningún modelo desde cero, identificando tanto sus fortalezas como sus limitaciones frente a casos de prueba controlados.

#### Escenario y datos utilizados

Un centro de soporte técnico recibe a diario comentarios de usuarios, tickets de solicitud, reportes de incidentes y documentación en inglés que debe ponerse a disposición del equipo hispanohablante. No se utiliza un dataset externo: se definen manualmente casos de prueba representativos (comentarios de usuarios, tickets de soporte, un reporte de incidente técnico y fragmentos de documentación en inglés) para evaluar cada modelo de forma controlada.

#### Estructura del notebook

| Fase | Contenido |
|---|---|
| 1 | Preparación del entorno (instalación de dependencias, detección de GPU/CPU) |
| 2 | Problema 1 – Análisis de sentimientos en comentarios de usuarios (español) |
| 3 | Problema 2 – Clasificación automática de tickets de soporte (zero-shot, multilingüe) |
| 4 | Problema 3 – Generación automática de resúmenes de incidentes técnicos (comparación abstractivo vs. extractivo) |
| 5 | Problema 4 – Traducción de documentación técnica de inglés a español |

#### Principales librerías

| Librería | Uso |
|---|---|
| `transformers` | Pipelines de Hugging Face: `sentiment-analysis`, `zero-shot-classification`, `summarization`, `text2text-generation`, `translation` |
| `sentence-transformers` | Embeddings semánticos multilingües para el resumen extractivo (Problema 3) |
| `torch` | Backend de inferencia de los modelos Transformer |
| `nltk` | Tokenización de oraciones (`sent_tokenize`) para el resumen extractivo |
| `scikit-learn` | Similitud coseno entre embeddings |
| `pandas` | Manipulación y presentación de resultados |


## 1. Preparación del entorno

Se utiliza el entorno conda con Python 3.10. Debido a la compatibilidad de los pipelines de `summarization` y `translation`, se fija una versión 4.x estable de `transformers`. El código detecta automáticamente si hay una GPU disponible (CUDA), pero funciona igualmente sobre CPU.

```
pip install "transformers<5.0.0" torch sentencepiece sacremoses emoji
```

In [49]:
# Ejecutar esta celda una sola vez si el entorno no tiene instaladas las dependencias.
!pip install -q "transformers<5.0.0" torch sentencepiece sacremoses emoji pandas

In [50]:
import warnings

import pandas as pd
import torch
import transformers
from transformers import pipeline

pd.set_option("display.max_colwidth", 200)

# El pipeline usará GPU (device=0) si CUDA está disponible; de lo contrario, CPU (device=-1).
DEVICE = 0 if torch.cuda.is_available() else -1

print("Versión de transformers:", transformers.__version__)
print("Versión de torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo utilizado:", "GPU" if DEVICE == 0 else "CPU")

Versión de transformers: 4.57.6
Versión de torch: 2.14.0+cpu
CUDA disponible: False
Dispositivo utilizado: CPU


## 2. Problema 1 – Análisis de sentimientos

**Contexto:** un sistema de soporte técnico recibe comentarios de los usuarios luego de resolver una incidencia. Se desea identificar automáticamente si la percepción expresada por el usuario es positiva, negativa o neutral, para priorizar seguimientos y medir satisfacción.

**Modelo:** [`pysentimiento/robertuito-sentiment-analysis`](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis), basado en RoBERTuito (una variante de RoBERTa preentrenada específicamente sobre texto en español de redes sociales). Resulta apropiado porque:
- Está afinado específicamente para **español**, el idioma en que los usuarios reportan sus comentarios.
- Clasifica en las tres categorías que necesitamos: `POS` (positivo), `NEG` (negativo) y `NEU` (neutral).
- Al estar entrenado sobre lenguaje informal/social, maneja razonablemente expresiones coloquiales similares a las que escriben los usuarios de soporte.

In [51]:
MODELO_SENTIMIENTO = "pysentimiento/robertuito-sentiment-analysis"

# Se dejan visibles los warnings de transformers (no se ocultan) para detectar problemas de compatibilidad.
warnings.filterwarnings("default", module="transformers")

clasificador_sentimiento = pipeline(
    "sentiment-analysis",
    model=MODELO_SENTIMIENTO,
    device=DEVICE,
)

# Etiquetas devueltas por el modelo -> nombre legible en español.
ETIQUETAS_SENTIMIENTO = {"POS": "Positivo", "NEG": "Negativo", "NEU": "Neutral"}


def analizar_sentimiento(texto: str) -> dict:
    """Clasifica un comentario en Positivo/Negativo/Neutral usando RoBERTuito.

    Devuelve un diccionario con la etiqueta legible y el score de confianza.
    """
    resultado = clasificador_sentimiento(texto)[0]
    return {
        "etiqueta": ETIQUETAS_SENTIMIENTO.get(resultado["label"], resultado["label"]),
        "confianza": round(resultado["score"], 4),
    }


# Prueba rápida de la función
analizar_sentimiento("El soporte resolvió mi problema muy rápido, excelente atención.")

Device set to use cpu


{'etiqueta': 'Positivo', 'confianza': 0.8544}

In [52]:
# Casos de prueba: 2 positivos, 2 negativos, 1 neutral y 1 ambiguo/mixto.
casos_prueba_sentimiento = [
    {
        "comentario": "El sistema funciona perfectamente y el soporte resolvió mi problema muy rápido.",
        "resultado_esperado": "Positivo",
    },
    {
        "comentario": "Excelente atención, el técnico fue muy amable y explicó todo con claridad.",
        "resultado_esperado": "Positivo",
    },
    {
        "comentario": "Llevo horas intentando ingresar y nadie me da una solución.",
        "resultado_esperado": "Negativo",
    },
    {
        "comentario": "Pésimo servicio, ya es la tercera vez que reporto el mismo error sin respuesta.",
        "resultado_esperado": "Negativo",
    },
    {
        "comentario": "La actualización fue instalada esta mañana.",
        "resultado_esperado": "Neutral",
    },
    {
        "comentario": "Finalmente solucionaron el problema, aunque el soporte tardó demasiado.",
        "resultado_esperado": "Ambiguo/Mixto",
    },
]

resultados_sentimiento = []
for caso in casos_prueba_sentimiento:
    prediccion = analizar_sentimiento(caso["comentario"])
    resultados_sentimiento.append(
        {
            "Comentario": caso["comentario"],
            "Resultado esperado": caso["resultado_esperado"],
            "Predicción": prediccion["etiqueta"],
            "Confianza": prediccion["confianza"],
        }
    )
    print(f"[{prediccion['etiqueta']:>8} | {prediccion['confianza']:.4f}] {caso['comentario']}")

[Positivo | 0.6690] El sistema funciona perfectamente y el soporte resolvió mi problema muy rápido.
[Positivo | 0.9633] Excelente atención, el técnico fue muy amable y explicó todo con claridad.
[Negativo | 0.9429] Llevo horas intentando ingresar y nadie me da una solución.
[Negativo | 0.9505] Pésimo servicio, ya es la tercera vez que reporto el mismo error sin respuesta.
[ Neutral | 0.8746] La actualización fue instalada esta mañana.
[ Neutral | 0.6283] Finalmente solucionaron el problema, aunque el soporte tardó demasiado.


In [53]:
df_sentimiento = pd.DataFrame(resultados_sentimiento)

df_sentimiento

,Comentario,Resultado esperado,Predicción,Confianza
0,El sistema funciona perfectamente y el soporte resolvió mi problema muy rápido.,Positivo,Positivo,0.6690
1,"Excelente atención, el técnico fue muy amable y explicó todo con claridad.",Positivo,Positivo,0.9633
2,Llevo horas intentando ingresar y nadie me da una solución.,Negativo,Negativo,0.9429
3,"Pésimo servicio, ya es la tercera vez que reporto el mismo error sin respuesta.",Negativo,Negativo,0.9505
4,La actualización fue instalada esta mañana.,Neutral,Neutral,0.8746
5,"Finalmente solucionaron el problema, aunque el soporte tardó demasiado.",Ambiguo/Mixto,Neutral,0.6283


### Análisis de resultados

Con los 6 casos de prueba, el modelo `pysentimiento/robertuito-sentiment-analysis` acertó en 5 de 6 comentarios frente al resultado esperado:

- **Positivos:** ambos comentarios positivos fueron clasificados correctamente como `Positivo`, aunque con confianza dispar (0.669 y 0.963). El primero ("funciona perfectamente... resolvió mi problema muy rápido") tiene menor confianza, posiblemente porque combina varias ideas dentro de una misma oración.
- **Negativos:** ambos comentarios negativos fueron clasificados correctamente como `Negativo`, con confianza alta (>0.94) en los dos casos, lo que sugiere que el modelo reconoce con solidez expresiones de queja o frustración.
- **Neutral:** el comentario neutral ("La actualización fue instalada esta mañana") fue clasificado correctamente como `Neutral` con alta confianza (0.875).
- **Caso ambiguo/mixto:** el comentario "Finalmente solucionaron el problema, aunque el soporte tardó demasiado" —que mezcla una parte positiva (solución) con una crítica (demora)— fue clasificado como `Neutral` con confianza moderada (0.628), la más baja del conjunto junto con el primer caso positivo. Esto es coherente con la naturaleza mixta del texto: el modelo no lo interpretó como claramente positivo ni negativo, sino que promedió ambas señales hacia el punto medio.

En síntesis, el modelo es robusto para sentimientos claramente polarizados (positivo/negativo), y su menor confianza se concentra precisamente en los comentarios con matices o mezcla de emociones, lo cual es esperable y útil para un sistema de soporte: la confianza baja puede usarse como señal para enviar esos casos a revisión humana.

## 3. Problema 2 – Clasificación automática de tickets

**Contexto:** un centro de soporte recibe cientos de solicitudes por distintos canales y necesita enrutarlas automáticamente al equipo correcto (Software, Hardware, Redes, Seguridad o Acceso y cuentas) sin depender de una clasificación manual.

**Modelo:** [`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7), usado mediante el pipeline `zero-shot-classification`. Resulta apropiado porque:
- Es **multilingüe** (entrenado sobre XNLI + otros datasets NLI en varios idiomas), por lo que puede clasificar tickets escritos en español sin necesidad de un modelo específico por idioma.
- Permite **zero-shot**: se definen las categorías en tiempo de ejecución (como hipótesis de entailment) sin necesidad de reentrenar el modelo para nuevas categorías.
- Al basarse en NLI (Natural Language Inference), estima qué tan bien cada categoría "se infiere" del texto del ticket, entregando un score de confianza por categoría.

In [54]:
MODELO_CLASIFICACION_TICKETS = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

CATEGORIAS_TICKETS = ["Software", "Hardware", "Redes", "Seguridad", "Acceso y cuentas"]

clasificador_tickets = pipeline(
    "zero-shot-classification",
    model=MODELO_CLASIFICACION_TICKETS,
    device=DEVICE,
)


def clasificar_ticket(texto: str, categorias: list = CATEGORIAS_TICKETS) -> dict:
    """Clasifica un ticket de soporte en una de las categorías dadas (zero-shot).

    Devuelve la categoría más probable, su score y el ranking completo de categorías.
    """
    resultado = clasificador_tickets(texto, candidate_labels=categorias)
    ranking = list(zip(resultado["labels"], [round(s, 4) for s in resultado["scores"]]))
    return {
        "categoria": resultado["labels"][0],
        "confianza": round(resultado["scores"][0], 4),
        "ranking": ranking,
    }


# Prueba rápida de la función
clasificar_ticket("El computador no reconoce el segundo monitor conectado por HDMI.")

Device set to use cpu


{'categoria': 'Hardware',
 'confianza': 0.5657,
 'ranking': [('Hardware', 0.5657),
  ('Software', 0.1478),
  ('Redes', 0.1315),
  ('Acceso y cuentas', 0.1082),
  ('Seguridad', 0.0467)]}

In [55]:
# 9 tickets de prueba: casos claros por categoría y algunos ambiguos.
casos_prueba_tickets = [
    {
        "ticket": "Después de actualizar la aplicación, el sistema se cierra cuando intento generar un reporte.",
        "categoria_esperada": "Software",
    },
    {
        "ticket": "El computador no reconoce el segundo monitor conectado por HDMI.",
        "categoria_esperada": "Hardware",
    },
    {
        "ticket": "La VPN se conecta pero no puedo acceder al servidor interno.",
        "categoria_esperada": "Redes",
    },
    {
        "ticket": "Recibí un correo solicitando ingresar mis credenciales en un enlace extraño.",
        "categoria_esperada": "Seguridad",
    },
    {
        "ticket": "Mi contraseña expiró y no puedo iniciar sesión.",
        "categoria_esperada": "Acceso y cuentas",
    },
    {
        "ticket": "El portátil no enciende desde que se le derramó agua encima.",
        "categoria_esperada": "Hardware",
    },
    {
        "ticket": "La aplicación de facturación muestra un error 500 al guardar un registro.",
        "categoria_esperada": "Software",
    },
    {
        "ticket": "Detecté un inicio de sesión desde una ubicación desconocida en mi cuenta corporativa.",
        "categoria_esperada": "Seguridad",
    },
    {
        "ticket": "El internet de la oficina se cae constantemente desde ayer.",
        "categoria_esperada": "Redes",
    },
]

resultados_tickets = []
for caso in casos_prueba_tickets:
    prediccion = clasificar_ticket(caso["ticket"])
    resultados_tickets.append(
        {
            "Ticket": caso["ticket"],
            "Categoría esperada": caso["categoria_esperada"],
            "Categoría predicha": prediccion["categoria"],
            "Confianza": prediccion["confianza"],
            "Ranking": prediccion["ranking"],
        }
    )
    print(f"[{prediccion['categoria']:>17} | {prediccion['confianza']:.4f}] {caso['ticket']}")

[         Software | 0.9541] Después de actualizar la aplicación, el sistema se cierra cuando intento generar un reporte.
[         Hardware | 0.5657] El computador no reconoce el segundo monitor conectado por HDMI.
[            Redes | 0.6564] La VPN se conecta pero no puedo acceder al servidor interno.
[ Acceso y cuentas | 0.5102] Recibí un correo solicitando ingresar mis credenciales en un enlace extraño.
[        Seguridad | 0.5531] Mi contraseña expiró y no puedo iniciar sesión.
[         Hardware | 0.7930] El portátil no enciende desde que se le derramó agua encima.
[         Software | 0.5666] La aplicación de facturación muestra un error 500 al guardar un registro.
[ Acceso y cuentas | 0.5427] Detecté un inicio de sesión desde una ubicación desconocida en mi cuenta corporativa.
[            Redes | 0.4575] El internet de la oficina se cae constantemente desde ayer.


In [56]:
df_tickets = pd.DataFrame(resultados_tickets)
df_tickets

,Ticket,Categoría esperada,Categoría predicha,Confianza,Ranking
0,"Después de actualizar la aplicación, el sistema se cierra cuando intento generar un reporte.",Software,Software,0.9541,"[(Software, 0.9541), (Acceso y cuentas, 0.0176), (Redes, 0.0146), (Hardware, 0.0075), (Seguridad, 0.0061)]"
1,El computador no reconoce el segundo monitor conectado por HDMI.,Hardware,Hardware,0.5657,"[(Hardware, 0.5657), (Software, 0.1478), (Redes, 0.1315), (Acceso y cuentas, 0.1082), (Seguridad, 0.0467)]"
2,La VPN se conecta pero no puedo acceder al servidor interno.,Redes,Redes,0.6564,"[(Redes, 0.6564), (Seguridad, 0.166), (Software, 0.0731), (Hardware, 0.0669), (Acceso y cuentas, 0.0375)]"
3,Recibí un correo solicitando ingresar mis credenciales en un enlace extraño.,Seguridad,Acceso y cuentas,0.5102,"[(Acceso y cuentas, 0.5102), (Seguridad, 0.2141), (Redes, 0.1444), (Hardware, 0.0732), (Software, 0.0581)]"
4,Mi contraseña expiró y no puedo iniciar sesión.,Acceso y cuentas,Seguridad,0.5531,"[(Seguridad, 0.5531), (Redes, 0.1982), (Acceso y cuentas, 0.1106), (Hardware, 0.0852), (Software, 0.0528)]"
5,El portátil no enciende desde que se le derramó agua encima.,Hardware,Hardware,0.7930,"[(Hardware, 0.793), (Redes, 0.1269), (Acceso y cuentas, 0.0365), (Software, 0.0219), (Seguridad, 0.0217)]"
6,La aplicación de facturación muestra un error 500 al guardar un registro.,Software,Software,0.5666,"[(Software, 0.5666), (Acceso y cuentas, 0.2827), (Redes, 0.1056), (Hardware, 0.0254), (Seguridad, 0.0197)]"
7,Detecté un inicio de sesión desde una ubicación desconocida en mi cuenta corporativa.,Seguridad,Acceso y cuentas,0.5427,"[(Acceso y cuentas, 0.5427), (Redes, 0.1626), (Seguridad, 0.117), (Hardware, 0.1002), (Software, 0.0775)]"
8,El internet de la oficina se cae constantemente desde ayer.,Redes,Redes,0.4575,"[(Redes, 0.4575), (Hardware, 0.1942), (Acceso y cuentas, 0.1428), (Seguridad, 0.1256), (Software, 0.0799)]"


### Análisis de resultados

Con los 9 tickets de prueba, el modelo acertó en 6 de 9 casos frente a la categoría esperada:

- **Software:** los dos tickets de software fueron clasificados correctamente, con confianza alta en el primero (0.9541, error tras actualización) y moderada en el segundo (0.5666, error 500 en facturación), donde "Acceso y cuentas" apareció como segunda opción con peso considerable (0.2827).
- **Hardware:** ambos tickets (monitor HDMI, portátil dañado por agua) fueron clasificados correctamente, con confianza más sólida en el caso físico evidente (0.793) que en el de conectividad de video (0.5657), donde "Software" y "Redes" también obtuvieron peso relevante.
- **Redes:** los dos tickets de red se clasificaron correctamente, aunque el segundo ("el internet se cae") tuvo la confianza más baja de todo el conjunto (0.4575), compitiendo de cerca con Hardware y Acceso y cuentas.
- **Seguridad vs. Acceso y cuentas:** los tres casos que cruzan estas dos categorías fallaron o se intercambiaron:
  - "Correo solicitando credenciales" (phishing, esperado `Seguridad`) → predicho `Acceso y cuentas` (0.5102).
  - "Contraseña expirada" (esperado `Acceso y cuentas`) → predicho `Seguridad` (0.5531).
  - "Inicio de sesión desde ubicación desconocida" (esperado `Seguridad`) → predicho `Acceso y cuentas` (0.5427).

  En los tres casos, la categoría correcta según nuestro criterio aparece como segunda opción en el ranking, con scores relativamente cercanos al primer lugar. Esto evidencia que **Seguridad** y **Acceso y cuentas** son conceptualmente solapadas en el escenario planteado (ambas giran en torno a credenciales/cuentas), y que el modelo zero-shot, al no haber sido afinado para esta taxonomía específica, distribuye probabilidad entre ambas de forma razonable pero no siempre coincidente con la etiqueta que nosotros esperábamos.

En resumen, el modelo funciona bien para categorías con vocabulario técnico distintivo (Software, Hardware, Redes), pero su desempeño baja cuando dos categorías del negocio comparten semántica (Seguridad/Acceso y cuentas), lo que sugiere que en un sistema real convendría fusionar esas categorías o añadir ejemplos de referencia (few-shot) para diferenciarlas mejor.

## 4. Problema 3 – Generación automática de resúmenes de incidentes técnicos

**Contexto:** los incidentes tecnológicos generan reportes técnicos extensos (bitácoras de análisis, causas, acciones correctivas). Revisarlos íntegramente consume tiempo, y en un contexto de soporte técnico la **fidelidad** del resumen es especialmente crítica: una alucinación puede introducir información incorrecta sobre causas, sistemas, fechas o soluciones, llevando a decisiones erradas.

Este problema se aborda en dos etapas:

1. **Enfoque abstractivo** (secciones 4.1 y 4.2): se prueban modelos generativos seq2seq (`mT5_multilingual_XLSum`, `flan-t5-base`) e incluso una estrategia map-reduce para superar la limitación de contexto. Como se documenta a continuación, estos experimentos presentaron alucinaciones o pérdida de información relevante, por lo que **se conservan como evidencia** de esta limitación.
2. **Enfoque extractivo con Transformer** (sección 4.3): dado que el enfoque abstractivo no resultó satisfactorio, se implementa un resumen **extractivo** basado en embeddings semánticos generados con un Transformer (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`). Este modelo no genera texto nuevo: se utiliza para representar semánticamente las oraciones del reporte y seleccionar, mediante similitud coseno, las que mejor cubren los elementos esenciales del incidente (problema, impacto, causa, acciones y solución).

**Modelo del primer intento (abstractivo):** [`csebuetnlp/mT5_multilingual_XLSum`](https://huggingface.co/csebuetnlp/mT5_multilingual_XLSum), una variante de mT5 afinada específicamente para la tarea de *summarization* sobre el dataset XL-Sum en 45 idiomas, incluido el español. Resulta apropiado porque:
- Es un modelo **secuencia a secuencia (seq2seq)** entrenado explícitamente para generar resúmenes abstractivos (no solo extraer oraciones), lo que le permite reformular el contenido.
- Soporta **español** de forma nativa, sin necesidad de traducir el texto antes de resumir.
- Al haber sido afinado sobre resúmenes periodísticos reales (XL-Sum), tiende a priorizar la idea central del texto, útil para condensar la esencia de un reporte de incidente.


In [57]:
reporte_incidente = """
El pasado martes a las 8:47 a.m., el equipo de monitoreo detectó que la aplicación web de gestión
de pedidos dejó de responder para la totalidad de los usuarios internos y externos. Los primeros
reportes indicaban tiempos de espera superiores a 30 segundos y, posteriormente, errores 503
de servicio no disponible en todas las peticiones. El equipo de soporte escaló el incidente a
severidad crítica dado que el módulo afectado procesa pedidos de clientes en tiempo real y su
indisponibilidad representaba una pérdida directa de ingresos para la compañía.

Durante la fase de análisis, el equipo de infraestructura revisó los tableros de monitoreo y
encontró que el servidor de aplicaciones presentaba un uso de CPU sostenido por encima del 95%
desde las 8:30 a.m., acompañado de un incremento anómalo en el número de conexiones abiertas hacia
la base de datos PostgreSQL. Al revisar los registros del motor de base de datos se identificó que
el pool de conexiones había alcanzado su límite máximo configurado (100 conexiones simultáneas),
provocando que nuevas solicitudes de la aplicación quedaran en espera indefinida y, finalmente,
expiraran por timeout. Adicionalmente, se detectó una consulta SQL sin índice adecuado sobre la
tabla de historial de pedidos, la cual comenzó a ejecutarse con mayor frecuencia tras el despliegue
de una nueva funcionalidad de reportes liberada la noche anterior.

La causa raíz identificada fue la combinación de dos factores: primero, la consulta sin optimizar
generaba bloqueos prolongados sobre la tabla de historial, aumentando el tiempo de respuesta de
cada transacción; segundo, el pool de conexiones de la aplicación no liberaba correctamente las
conexiones asociadas a esas transacciones lentas, agotando los recursos disponibles y bloqueando al
resto de los usuarios.

Como acción correctiva inmediata, el equipo de guardia reinició el servicio de aplicaciones para
liberar las conexiones colgadas y restableció el acceso de forma temporal. En paralelo, el equipo de
base de datos creó un índice sobre la columna utilizada en la nueva consulta de reportes, reduciendo
su tiempo de ejecución de más de 12 segundos a menos de 200 milisegundos. También se ajustó el
tamaño máximo del pool de conexiones y se configuró un tiempo de espera (timeout) más estricto para
evitar que conexiones lentas bloquearan el resto del sistema.

El servicio quedó completamente restablecido a las 10:15 a.m., con un tiempo total de indisponibilidad
de aproximadamente 88 minutos. Como acción de seguimiento, se programó una revisión del proceso de
despliegue para incluir pruebas de carga obligatorias antes de liberar nuevas funcionalidades que
impacten consultas sobre tablas de alto volumen.
""".strip()

palabras_originales = len(reporte_incidente.split())
print("Palabras del reporte original:", palabras_originales)
print("\n" + reporte_incidente)

Palabras del reporte original: 421

El pasado martes a las 8:47 a.m., el equipo de monitoreo detectó que la aplicación web de gestión
de pedidos dejó de responder para la totalidad de los usuarios internos y externos. Los primeros
reportes indicaban tiempos de espera superiores a 30 segundos y, posteriormente, errores 503
de servicio no disponible en todas las peticiones. El equipo de soporte escaló el incidente a
severidad crítica dado que el módulo afectado procesa pedidos de clientes en tiempo real y su
indisponibilidad representaba una pérdida directa de ingresos para la compañía.

Durante la fase de análisis, el equipo de infraestructura revisó los tableros de monitoreo y
encontró que el servidor de aplicaciones presentaba un uso de CPU sostenido por encima del 95%
desde las 8:30 a.m., acompañado de un incremento anómalo en el número de conexiones abiertas hacia
la base de datos PostgreSQL. Al revisar los registros del motor de base de datos se identificó que
el pool de conexiones

In [58]:
MODELO_RESUMEN = "csebuetnlp/mT5_multilingual_XLSum"

resumidor = pipeline(
    "summarization",
    model=MODELO_RESUMEN,
    tokenizer=MODELO_RESUMEN,
    device=DEVICE,
)


def resumir_incidente(texto: str, max_length: int = 84, min_length: int = 20) -> str:
    """Genera un resumen abstractivo del reporte de incidente con mT5_multilingual_XLSum (configuración por defecto)."""
    resultado = resumidor(texto, max_length=max_length, min_length=min_length, do_sample=False)
    return resultado[0]["summary_text"]


resumen_generado = resumir_incidente(reporte_incidente)
palabras_resumen = len(resumen_generado.split())
reduccion = (1 - palabras_resumen / palabras_originales) * 100

print("Resumen generado (configuración por defecto):\n", resumen_generado)
print("\nPalabras originales:", palabras_originales)
print("Palabras del resumen:", palabras_resumen)
print(f"Porcentaje de reducción: {reduccion:.1f}%")

c:\Users\ing_c\miniconda3\envs\tf-gpu\lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cpu


Resumen generado (configuración por defecto):
 El equipo de guardia de la compañía Microsoft informó este miércoles que el sistema operativo de sus aplicaciones se encuentra en estado crítico después de que se detectaran errores en los tableros de monitoreo.

Palabras originales: 421
Palabras del resumen: 34
Porcentaje de reducción: 91.9%


### Ajuste según recomendaciones del model card

El [model card](https://huggingface.co/csebuetnlp/mT5_multilingual_XLSum) de `mT5_multilingual_XLSum` recomienda dos ajustes frente a la configuración por defecto usada arriba:

1. **Limpieza del texto de entrada** (`WHITESPACE_HANDLER`): colapsar saltos de línea y espacios múltiples, ya que el modelo fue entrenado sobre artículos en una sola línea continua.
2. **Beam search en vez de greedy decoding**: `num_beams=4` y `no_repeat_ngram_size=2`, en lugar de la generación greedy (beam=1) usada por defecto en el pipeline.

Se repite la generación del resumen aplicando ambos ajustes, para comparar contra el resultado anterior.

In [59]:
import re

# Limpieza recomendada por el model card: colapsar saltos de línea y espacios múltiples.
WHITESPACE_HANDLER = lambda texto: re.sub(r"\s+", " ", re.sub(r"\n+", " ", texto.strip()))


def resumir_incidente_mejorado(texto: str, max_length: int = 84) -> str:
    """Genera el resumen aplicando limpieza de texto y beam search (recomendación del model card)."""
    texto_limpio = WHITESPACE_HANDLER(texto)
    resultado = resumidor(
        texto_limpio,
        max_length=max_length,
        num_beams=4,
        no_repeat_ngram_size=2,
        do_sample=False,
    )
    return resultado[0]["summary_text"]


resumen_mejorado = resumir_incidente_mejorado(reporte_incidente)
palabras_resumen_mejorado = len(resumen_mejorado.split())
reduccion_mejorada = (1 - palabras_resumen_mejorado / palabras_originales) * 100

print("Resumen generado (limpieza + beam search):\n", resumen_mejorado)
print("\nPalabras originales:", palabras_originales)
print("Palabras del resumen mejorado:", palabras_resumen_mejorado)
print(f"Porcentaje de reducción: {reduccion_mejorada:.1f}%")

Resumen generado (limpieza + beam search):
 El equipo de guardia de la compañía Microsoft informó este miércoles que el sistema operativo de sus aplicaciones se encuentra en estado crítico después de que se detectaran errores en los tableros de monitoreo.

Palabras originales: 421
Palabras del resumen mejorado: 34
Porcentaje de reducción: 91.9%


### Análisis de resultados

El resumen con configuración por defecto y el resumen con las mejoras recomendadas por el model card (limpieza de saltos de línea + `num_beams=4` + `no_repeat_ngram_size=2`) resultaron **idénticos**:

> "El equipo de guardia de la compañía Microsoft informó este miércoles que el sistema operativo de sus aplicaciones se encuentra en estado crítico después de que se detectaran errores en los tableros de monitoreo."

Esto es un resultado relevante: para este texto de entrada, ni limpiar el formato ni cambiar de *greedy decoding* a *beam search* modificó la salida del modelo. Esto indica que el problema **no está en los parámetros de generación ni en el formato del texto**, sino en el propio modelo y en el desajuste de dominio: `mT5_multilingual_XLSum` fue afinado sobre resúmenes de **noticias periodísticas** (BBC/XL-Sum), y frente a un texto con vocabulario técnico (PostgreSQL, pool de conexiones, timeout, índice de base de datos) tiende a "encajarlo" dentro del patrón narrativo que aprendió (una entidad, un vocero, un día de la semana), incluso inventando esos elementos cuando no existen en el texto original.

Evaluación cualitativa frente al reporte original (aplica a ambas configuraciones):

- **Problema presentado:** parcialmente conservado (estado crítico, tableros de monitoreo), mas no el hecho concreto de que la aplicación web de pedidos dejó de responder.
- **Impacto:** **no conservado** (no menciona los 88 minutos de indisponibilidad ni la pérdida de ingresos).
- **Causa:** **no conservada** (no aparece PostgreSQL, el pool de conexiones agotado ni la consulta SQL sin índice).
- **Solución:** **no conservada** (no aparece el reinicio del servicio, la creación del índice ni el ajuste del pool).
- **Alucinación:** persiste en ambas configuraciones ("compañía Microsoft", "este miércoles", "sistema operativo"), confirmando que no era un artefacto de la decodificación greedy sino un patrón aprendido del dominio de entrenamiento.

**Conclusión de la mejora aplicada:** ajustar los parámetros de generación según las recomendaciones oficiales del modelo **no resolvió el problema de fidelidad**, lo cual es una limitación real y valiosa de documentar. Para este caso de uso específico (resumen de reportes técnicos de incidentes), sería necesario **afinar (fine-tuning) el modelo con reportes de incidentes reales**, o usar un modelo de resumen extractivo (que solo selecciona oraciones del texto original) en lugar de uno abstractivo, para garantizar que no se inventen datos. Mientras tanto, este modelo solo debería usarse como borrador inicial sujeto a revisión humana obligatoria.

### Registro de intentos realizados (Problema 3)

Antes de continuar, se resume lo probado hasta ahora con `csebuetnlp/mT5_multilingual_XLSum`:

| Intento | Configuración | Resultado |
|---|---|---|
| 1 | Pipeline `summarization` por defecto (`max_length=84`, `min_length=20`, greedy decoding) | Resumen de 34 palabras (91.9% de reducción) que **alucina** datos no presentes en el texto ("compañía Microsoft", "este miércoles") y **no conserva** causa, impacto ni solución. |
| 2 | Preprocesamiento recomendado por el model card (`WHITESPACE_HANDLER`) + `num_beams=4` + `no_repeat_ngram_size=2` | Resultado **idéntico** al intento 1: mismo texto alucinado, misma pérdida de información clave. |

**Conclusión parcial:** el problema no está en los parámetros de generación ni en el formato del texto de entrada, sino en el desajuste entre el dominio de entrenamiento del modelo (noticias periodísticas de XL-Sum) y el dominio del texto a resumir (reporte técnico de incidentes). A continuación se prueba un modelo distinto para verificar si un enfoque diferente entrega mejores resultados.

### 4.1 Prueba adicional con `google/flan-t5-base`

[`google/flan-t5-base`](https://huggingface.co/google/flan-t5-base) es un modelo T5 ajustado por instrucciones (*instruction-tuned*) sobre una gran variedad de tareas, incluyendo resumen. A diferencia de `mT5_multilingual_XLSum`, no fue afinado específicamente sobre noticias periodísticas, sino sobre instrucciones en lenguaje natural (p. ej. `"Summarize: <texto>"`), lo que en teoría podría generar resúmenes más apegados al contenido real en lugar de imitar un estilo narrativo fijo.

**Limitación a tener en cuenta:** `flan-t5-base` es un modelo predominantemente **entrenado y evaluado en inglés**; su soporte de español es limitado en comparación con un modelo multilingüe como mT5. Se prueba de todas formas para verificar empíricamente su comportamiento sobre el mismo reporte técnico en español.

In [60]:
MODELO_RESUMEN_ALTERNATIVO = "google/flan-t5-base"

resumidor_flan = pipeline(
    "text2text-generation",
    model=MODELO_RESUMEN_ALTERNATIVO,
    device=DEVICE,
)


def resumir_incidente_flan(texto: str, max_new_tokens: int = 84) -> str:
    """Genera un resumen usando flan-t5-base mediante una instrucción explícita en el prompt."""
    prompt = f"Summarize the following technical incident report: {texto}"
    resultado = resumidor_flan(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return resultado[0]["generated_text"]


resumen_flan = resumir_incidente_flan(reporte_incidente)
palabras_resumen_flan = len(resumen_flan.split())
reduccion_flan = (1 - palabras_resumen_flan / palabras_originales) * 100

print("Resumen generado (flan-t5-base):\n", resumen_flan)
print("\nPalabras originales:", palabras_originales)
print("Palabras del resumen:", palabras_resumen_flan)
print(f"Porcentaje de reducción: {reduccion_flan:.1f}%")

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (1081 > 512). Running this sequence through the model will result in indexing errors


Resumen generado (flan-t5-base):
 El pasado martes a las 8:47 a.m., el equipo de monitoreo detectó que el web de gestión de pedidos dejó responder para todos los usuarios internos y externos. Los primeros reportes indicaban tiempo superiores a

Palabras originales: 421
Palabras del resumen: 35
Porcentaje de reducción: 91.7%


### Análisis de resultados — `flan-t5-base`

El resultado real fue el siguiente:

> "El pasado martes a las 8:47 a.m., el equipo de monitoreo detectó que el web de gestión de pedidos dejó responder para todos los usuarios internos y externos. Los primeros reportes indicaban tiempo superiores a"

Con 35 palabras (91.7% de reducción), a primera vista el número es similar al de `mT5_multilingual_XLSum`, pero el comportamiento es cualitativamente distinto y con sus propias limitaciones:

- **No es un resumen, es una copia truncada:** `flan-t5-base` no reformuló ni condensó el contenido; simplemente **reprodujo casi textualmente las primeras dos oraciones del reporte original** y se cortó a la mitad de una frase ("tiempo superiores a"). No hubo alucinación de datos falsos (a diferencia de mT5), pero tampoco hubo síntesis real de la información.
- **Advertencia de truncamiento de entrada:** la ejecución mostró el warning `Token indices sequence length is longer than the specified maximum sequence length for this model (1081 > 512)`, es decir, el reporte completo (421 palabras) **excede la ventana de contexto de 512 tokens** de `flan-t5-base` y el modelo solo "vio" una fracción inicial del texto. Esto explica por qué el resumen solo contiene información de las primeras líneas: **nunca llegó a procesar la causa, el impacto ni la solución**, que están en la segunda mitad del reporte.
- **Problema/causa/impacto/solución:** ninguno de los cuatro elementos se conserva de forma útil; solo aparece el inicio del problema (indisponibilidad reportada), sin desarrollo alguno.

**Comparación con `mT5_multilingual_XLSum`:**

| Aspecto | `mT5_multilingual_XLSum` | `flan-t5-base` |
|---|---|---|
| Estilo de salida | Abstractivo (reformula) | Prácticamente extractivo/copia literal |
| Alucinaciones | Sí (inventa entidades y fechas) | No inventa datos, pero tampoco resume |
| Cobertura del texto | Procesa el reporte completo | Trunca en ~512 tokens, ignora la segunda mitad |
| Utilidad para este caso | Baja (resumen fluido pero falso) | Baja (fiel pero incompleto, no es un resumen real) |

**Conclusión de la prueba adicional:** cambiar de modelo **no resolvió el problema de fondo**, solo cambió el tipo de falla: `mT5_multilingual_XLSum` alucina para producir un resumen "con forma", mientras que `flan-t5-base` es fiel al texto pero ni siquiera logra ver el reporte completo por la limitación de su ventana de contexto, y termina copiando en vez de resumir. Para este caso de uso específico, ninguno de los dos modelos preentrenados "tal cual" es suficiente sin ajustes adicionales (p. ej. dividir el texto en fragmentos que quepan en la ventana de contexto, o afinar el modelo con reportes de incidentes reales).

### 4.2 Estrategia map-reduce para resumir el reporte completo

Las dos pruebas anteriores mostraron problemas distintos: `mT5_multilingual_XLSum` alucina al resumir el texto completo de una sola vez, y `flan-t5-base` ni siquiera alcanza a "ver" la segunda mitad del reporte porque su ventana de contexto (512 tokens) es menor que la longitud del texto (1081 tokens).

Para atacar directamente esta limitación de longitud, se implementa una estrategia **map-reduce**, muy usada en resumen de documentos largos con modelos de contexto limitado:

1. **Map:** el reporte se divide en fragmentos más cortos (por número de palabras) y se genera un resumen independiente de cada fragmento con `mT5_multilingual_XLSum` (el mismo modelo/pipeline `resumidor` ya cargado, con los parámetros de beam search usados en el intento "mejorado").
2. **Reduce:** los resúmenes parciales se concatenan en un solo texto, y ese texto combinado se vuelve a resumir con el mismo modelo para producir el resumen final.

De esta forma, ningún fragmento individual excede la ventana de contexto del modelo, y el resumen final sí tiene la oportunidad de integrar información de todo el reporte (inicio, causa, impacto y solución), a diferencia de los dos intentos anteriores.

In [61]:
def dividir_en_fragmentos(texto: str, palabras_por_fragmento: int = 100) -> list:
    """Divide el texto en fragmentos consecutivos de tamaño similar (por número de palabras)."""
    palabras = texto.split()
    return [
        " ".join(palabras[i:i + palabras_por_fragmento])
        for i in range(0, len(palabras), palabras_por_fragmento)
    ]


def resumir_mapa(fragmentos: list, max_length: int = 60, min_length: int = 15) -> list:
    """Fase MAP: genera un resumen independiente para cada fragmento."""
    resumenes_parciales = []
    for fragmento in fragmentos:
        fragmento_limpio = WHITESPACE_HANDLER(fragmento)
        salida = resumidor(
            fragmento_limpio,
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            no_repeat_ngram_size=2,
            do_sample=False,
        )
        resumenes_parciales.append(salida[0]["summary_text"])
    return resumenes_parciales


def resumir_reduce(resumenes_parciales: list, max_length: int = 84, min_length: int = 20) -> str:
    """Fase REDUCE: combina los resúmenes parciales y genera el resumen final."""
    texto_combinado = WHITESPACE_HANDLER(" ".join(resumenes_parciales))
    salida = resumidor(
        texto_combinado,
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        no_repeat_ngram_size=2,
        do_sample=False,
    )
    return salida[0]["summary_text"]


def resumir_map_reduce(texto: str, palabras_por_fragmento: int = 100) -> dict:
    """Orquesta las fases map y reduce sobre el texto completo."""
    fragmentos = dividir_en_fragmentos(texto, palabras_por_fragmento)
    resumenes_parciales = resumir_mapa(fragmentos)
    resumen_final = resumir_reduce(resumenes_parciales)
    return {
        "fragmentos": fragmentos,
        "resumenes_parciales": resumenes_parciales,
        "resumen_final": resumen_final,
    }


resultado_map_reduce = resumir_map_reduce(reporte_incidente)
palabras_resumen_map_reduce = len(resultado_map_reduce["resumen_final"].split())
reduccion_map_reduce = (1 - palabras_resumen_map_reduce / palabras_originales) * 100

print(f"Número de fragmentos generados: {len(resultado_map_reduce['fragmentos'])}\n")
for i, (fragmento, resumen) in enumerate(
    zip(resultado_map_reduce["fragmentos"], resultado_map_reduce["resumenes_parciales"]), start=1
):
    print(f"--- Fragmento {i} ({len(fragmento.split())} palabras) ---")
    print(f"Resumen parcial: {resumen}\n")

print("Resumen final (map-reduce):\n", resultado_map_reduce["resumen_final"])
print("\nPalabras originales:", palabras_originales)
print("Palabras del resumen final:", palabras_resumen_map_reduce)
print(f"Porcentaje de reducción: {reduccion_map_reduce:.1f}%")

Your max_length is set to 60, but your input_length is only 40. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)


Número de fragmentos generados: 5

--- Fragmento 1 (100 palabras) ---
Resumen parcial: El equipo de soporte de la compañía Associated Press informó que la aplicación web de Gestión de pedidos de los clientes dejó de responder en tiempo real para la totalidad de sus usuarios internos y externos.

--- Fragmento 2 (100 palabras) ---
Resumen parcial: El servidor de aplicaciones de la Universidad de Cambridge (MIT) tuvo que revisar los registros de su sistema operativo y detectar que el número de solicitudes que se han solicitado no alcanzó su límite máximo.

--- Fragmento 3 (100 palabras) ---
Resumen parcial: El equipo de guardia de la aplicación Android informó este miércoles que la consulta de historial de los usuarios y sus datos puede ser ejecutada sin éxito.

--- Fragmento 4 (100 palabras) ---
Resumen parcial: El servicio de acceso a internet en Reino Unido fue restablecido de forma temporal este miércoles, según un nuevo estudio.

--- Fragmento 5 (21 palabras) ---
Resumen parcial: El

### Análisis de resultados — map-reduce

El resultado real fue el siguiente:

**Fase MAP (5 fragmentos de ~100 palabras):**

| Fragmento | Resumen parcial generado |
|---|---|
| 1 | "El equipo de soporte de la compañía **Associated Press** informó que la aplicación web de Gestión de pedidos de los clientes dejó de responder en tiempo real..." |
| 2 | "El servidor de aplicaciones de la **Universidad de Cambridge (MIT)** tuvo que revisar los registros de su sistema operativo..." |
| 3 | "El equipo de guardia de la **aplicación Android** informó este miércoles que la consulta de historial..." |
| 4 | "El servicio de acceso a internet en **Reino Unido** fue restablecido de forma temporal este miércoles, según un nuevo estudio." |
| 5 | "El **gobierno de Estados Unidos** anunció este miércoles el inicio de una nueva etapa de la **carga de gases** en el país." |

**Fase REDUCE (resumen final):**

> "El servicio de acceso a internet en Reino Unido fue restablecido de forma temporal este miércoles."

(16 palabras, 96.2% de reducción)

**Esta prueba no mejoró el problema, lo empeoró:** en vez de una alucinación por resumen, se obtuvieron **cinco alucinaciones distintas**, una por cada fragmento — el modelo inventó entidades completamente ajenas al texto (Associated Press, Universidad de Cambridge, MIT, Android, Reino Unido, Estados Unidos, "carga de gases"), ninguna de las cuales aparece en el reporte original. Al aplicar el REDUCE sobre estos resúmenes ya corrompidos, el resultado final terminó siendo **una sola oración irrelevante** ("el servicio de acceso a internet en Reino Unido fue restablecido"), que no tiene ninguna relación con el incidente real (caída de la app de gestión de pedidos por agotamiento del pool de conexiones a PostgreSQL).

**Diagnóstico:** `mT5_multilingual_XLSum` fue entrenado específicamente sobre artículos periodísticos de la BBC (XLSum), por lo que su comportamiento por defecto ante *cualquier* fragmento de texto, sin importar el dominio, es "redactarlo como si fuera una noticia" — completando con nombres de organizaciones, países y fechas típicas de titulares de prensa, en lugar de extraer los hechos técnicos reales. Dividir el texto en fragmentos más pequeños no elimina este sesgo del modelo; simplemente lo aplica repetidas veces (una alucinación distinta por fragmento) y luego el paso REDUCE combina y resume esas alucinaciones, degradando aún más la fidelidad.

**Comparación de los cuatro intentos sobre el Problema 3:**

| Intento | Modelo | Alucina | Cobertura del texto completo | Utilidad |
|---|---|---|---|---|
| Directo | mT5_multilingual_XLSum (greedy) | Sí | Sí | Baja |
| Beam search + limpieza | mT5_multilingual_XLSum (beam) | Sí (idéntico al anterior) | Sí | Baja |
| Prompt directo | flan-t5-base | No, pero copia literal y se trunca | No (límite 512 tokens) | Baja |
| Map-reduce | mT5_multilingual_XLSum (por fragmentos) | Sí, y en mayor cantidad | Sí, pero corrompida por fragmento | Muy baja |

**Conclusión general del Problema 3:** el cuello de botella no es la estrategia de resumen (directo, con mejores hiperparámetros, o map-reduce), sino el **desajuste de dominio del modelo preentrenado**: `mT5_multilingual_XLSum` está afinado para noticias y tiende a alucinar sistemáticamente frente a texto técnico de soporte, sin importar cómo se lo fragmente o qué parámetros de generación se usen; `flan-t5-base`, por su parte, es fiel pero no está afinado para resumir y además tiene una ventana de contexto insuficiente. Una solución robusta para este caso de uso requeriría **afinar (fine-tuning) un modelo con reportes de incidentes reales**, o usar un modelo de resumen de contexto largo (p. ej. `longformer`/`led-base`) entrenado en dominios técnicos, lo cual excede el alcance de esta actividad basada en modelos preentrenados "tal cual".

### 4.3 Segundo enfoque: resumen extractivo con un Transformer de embeddings semánticos

Dado que los intentos abstractivos anteriores (directo, con beam search, con `flan-t5-base` y con map-reduce) alucinaron información o perdieron contenido esencial del reporte, se implementa a continuación un **resumen extractivo**: en lugar de generar texto nuevo, se seleccionan las oraciones *ya existentes* en el reporte original que mejor representan sus elementos clave.

**Modelo:** [`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). Es importante precisar que **no es un modelo de summarization**: es un Transformer multilingüe utilizado para generar representaciones semánticas (embeddings) de las oraciones del reporte, las cuales posteriormente se utilizan para construir el resumen extractivo mediante similitud coseno. Tampoco fue entrenado específicamente para soporte técnico; se usa aquí como codificador semántico de propósito general.

**Procedimiento:**

1. Dividir el reporte técnico en oraciones (`dividir_en_oraciones`).
2. Generar embeddings de cada oración y de un conjunto de criterios semánticos que representan los elementos esperados de un incidente (`generar_embeddings`).
3. Calcular la similitud coseno entre cada oración y cada criterio.
4. Seleccionar las oraciones con mayor similitud máxima frente a los criterios (`seleccionar_oraciones_relevantes`), descartando oraciones muy cortas o duplicadas.
5. Reordenar las oraciones seleccionadas según su posición original en el texto, para conservar la coherencia temporal, y construir el resumen final (`generar_resumen_extractivo`).

Al utilizar exclusivamente oraciones tomadas literalmente del reporte original, este enfoque reduce el riesgo de que se introduzca información inexistente, aunque no garantiza que el resultado sea tan fluido como un resumen abstractivo.


In [62]:
!pip install -q sentence-transformers scikit-learn nltk

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


True

In [63]:
import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE_EMBEDDINGS = "cuda" if torch.cuda.is_available() else "cpu"

modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS, device=DEVICE_EMBEDDINGS)

# Criterios semánticos que representan los elementos esperados de un incidente de soporte.
criterios_resumen = [
    "descripción del problema o incidente técnico",
    "impacto del incidente sobre los usuarios o servicios",
    "causa identificada del problema",
    "acciones técnicas realizadas para solucionar el incidente",
    "solución aplicada y restablecimiento del servicio",
]
NOMBRES_CRITERIOS = ["problema", "impacto", "causa", "acciones", "solucion"]


def dividir_en_oraciones(texto: str) -> list:
    """Divide el reporte en oraciones; usa una expresión regular como respaldo si NLTK falla."""
    try:
        oraciones = nltk.sent_tokenize(texto, language="spanish")
    except LookupError:
        oraciones = re.split(r"(?<=[.!?])\s+", texto)
    return [o.strip() for o in oraciones if o.strip()]


def generar_embeddings(textos: list):
    """Genera embeddings normalizados (Transformer MiniLM) para una lista de oraciones o criterios."""
    return modelo_embeddings.encode(textos, normalize_embeddings=True)


def seleccionar_oraciones_relevantes(oraciones: list, criterios: list, n_oraciones: int = 5, min_palabras: int = 6):
    """Calcula la similitud coseno de cada oración con los criterios y selecciona las más relevantes.

    Devuelve el DataFrame de análisis completo y los índices seleccionados, ordenados por posición original.
    """
    embeddings_oraciones = generar_embeddings(oraciones)
    embeddings_criterios = generar_embeddings(criterios)
    similitud = cosine_similarity(embeddings_oraciones, embeddings_criterios)

    filas = []
    for i, oracion in enumerate(oraciones):
        fila = {"indice": i, "oracion": oracion}
        for nombre, score in zip(NOMBRES_CRITERIOS, similitud[i]):
            fila[f"score_{nombre}"] = round(float(score), 4)
        fila["score_maximo"] = round(float(similitud[i].max()), 4)
        filas.append(fila)

    df_analisis = pd.DataFrame(filas)

    # Se descartan oraciones muy cortas/poco informativas y duplicadas antes de seleccionar.
    candidatas = df_analisis[df_analisis["oracion"].str.split().str.len() >= min_palabras]
    candidatas = candidatas.drop_duplicates(subset="oracion")
    indices_seleccionados = (
        candidatas.sort_values("score_maximo", ascending=False).head(n_oraciones)["indice"].tolist()
    )

    df_analisis["seleccionada"] = df_analisis["indice"].isin(indices_seleccionados)
    return df_analisis, sorted(indices_seleccionados)


def generar_resumen_extractivo(texto: str, criterios: list, n_oraciones: int = 5):
    """Construye un resumen extractivo seleccionando y reordenando las oraciones más relevantes del texto."""
    oraciones = dividir_en_oraciones(texto)
    df_analisis, indices_seleccionados = seleccionar_oraciones_relevantes(oraciones, criterios, n_oraciones)
    resumen = " ".join(oraciones[i] for i in indices_seleccionados)
    return resumen, df_analisis


print("Modelo de embeddings cargado:", MODELO_EMBEDDINGS)
print("Dispositivo utilizado:", DEVICE_EMBEDDINGS)


Modelo de embeddings cargado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dispositivo utilizado: cpu


In [64]:
resumen_extractivo, df_analisis_oraciones = generar_resumen_extractivo(
    reporte_incidente, criterios_resumen, n_oraciones=5
)

palabras_resumen_extractivo = len(resumen_extractivo.split())
reduccion_extractiva = (1 - palabras_resumen_extractivo / palabras_originales) * 100

print("Resumen extractivo generado:\n", resumen_extractivo)
print("\nPalabras originales:", palabras_originales)
print("Palabras del resumen extractivo:", palabras_resumen_extractivo)
print(f"Porcentaje de reducción: {reduccion_extractiva:.1f}%")


Resumen extractivo generado:
 El pasado martes a las 8:47 a.m., el equipo de monitoreo detectó que la aplicación web de gestión
de pedidos dejó de responder para la totalidad de los usuarios internos y externos. El equipo de soporte escaló el incidente a
severidad crítica dado que el módulo afectado procesa pedidos de clientes en tiempo real y su
indisponibilidad representaba una pérdida directa de ingresos para la compañía. La causa raíz identificada fue la combinación de dos factores: primero, la consulta sin optimizar
generaba bloqueos prolongados sobre la tabla de historial, aumentando el tiempo de respuesta de
cada transacción; segundo, el pool de conexiones de la aplicación no liberaba correctamente las
conexiones asociadas a esas transacciones lentas, agotando los recursos disponibles y bloqueando al
resto de los usuarios. Como acción correctiva inmediata, el equipo de guardia reinició el servicio de aplicaciones para
liberar las conexiones colgadas y restableció el acceso de fo

**DataFrame de análisis de oraciones:** a continuación se muestran todas las oraciones del reporte con su score de similitud frente a cada criterio (`score_problema`, `score_impacto`, `score_causa`, `score_acciones`, `score_solucion`), su `score_maximo` y si fueron `seleccionada`s para el resumen final. Esto permite evidenciar por qué cada oración fue (o no) incluida.


In [65]:
df_analisis_oraciones.sort_values("score_maximo", ascending=False)


,indice,oracion,score_problema,score_impacto,score_causa,score_acciones,score_solucion,score_maximo,seleccionada
7,7,"Como acción correctiva inmediata, el equipo de guardia reinició el servicio de aplicaciones para\nliberar las conexiones colgadas y restableció el acceso de forma temporal.",0.2005,0.3391,0.0731,0.3061,0.5894,0.5894,True
2,2,El equipo de soporte escaló el incidente a\nseveridad crítica dado que el módulo afectado procesa pedidos de clientes en tiempo real y su\nindisponibilidad representaba una pérdida directa de ingr...,0.4953,0.5541,0.4137,0.3895,0.4450,0.5541,True
6,6,"La causa raíz identificada fue la combinación de dos factores: primero, la consulta sin optimizar\ngeneraba bloqueos prolongados sobre la tabla de historial, aumentando el tiempo de respuesta de\n...",0.3902,0.5121,0.5450,0.2768,0.3506,0.5450,True
0,0,"El pasado martes a las 8:47 a.m., el equipo de monitoreo detectó que la aplicación web de gestión\nde pedidos dejó de responder para la totalidad de los usuarios internos y externos.",0.2827,0.5225,0.1969,0.1927,0.3656,0.5225,True
10,10,"El servicio quedó completamente restablecido a las 10:15 a.m., con un tiempo total de indisponibilidad\nde aproximadamente 88 minutos.",0.1694,0.2471,-0.0195,0.1736,0.4902,0.4902,True
3,3,"Durante la fase de análisis, el equipo de infraestructura revisó los tableros de monitoreo y\nencontró que el servidor de aplicaciones presentaba un uso de CPU sostenido por encima del 95%\ndesde ...",0.3043,0.3591,0.2611,0.1839,0.2965,0.3591,False
11,11,"Como acción de seguimiento, se programó una revisión del proceso de\ndespliegue para incluir pruebas de carga obligatorias antes de liberar nuevas funcionalidades que\nimpacten consultas sobre tab...",0.3174,0.3107,0.2266,0.3425,0.2629,0.3425,False
1,1,"Los primeros\nreportes indicaban tiempos de espera superiores a 30 segundos y, posteriormente, errores 503\nde servicio no disponible en todas las peticiones.",0.3173,0.3250,0.1901,0.2098,0.2517,0.3250,False
5,5,"Adicionalmente, se detectó una consulta SQL sin índice adecuado sobre la\ntabla de historial de pedidos, la cual comenzó a ejecutarse con mayor frecuencia tras el despliegue\nde una nueva funciona...",0.2339,0.3211,0.2296,0.1637,0.2521,0.3211,False
4,4,"Al revisar los registros del motor de base de datos se identificó que\nel pool de conexiones había alcanzado su límite máximo configurado (100 conexiones simultáneas),\nprovocando que nuevas solic...",0.2737,0.2996,0.2209,0.1588,0.2789,0.2996,False


### Comparación: enfoque abstractivo vs. enfoque extractivo

| Criterio | Resumen abstractivo (mT5 XLSum) | Resumen extractivo (MiniLM + similitud coseno) |
|---|---|---|
| Genera texto nuevo | Sí (reformula/parafrasea) | No (solo reutiliza oraciones del reporte original) |
| Fidelidad al texto original | Baja: no conservó impacto, causa ni solución reales | Alta: las 5 oraciones seleccionadas provienen literalmente del reporte |
| Presencia de alucinaciones | Sí, en todas las variantes probadas (entidades y hechos inventados) | No aplica; al no generar texto nuevo, no puede inventar hechos |
| Fluidez | Alta (oración única, bien formada) | Media: el resumen es una concatenación de 5 oraciones extensas, por lo que resulta menos fluido y más largo |
| Trazabilidad | Baja (no es posible saber de dónde salió cada afirmación) | Alta: cada oración del resumen es identificable en el DataFrame de análisis, con su score por criterio |
| Adecuación al contexto de soporte técnico | Baja para este caso (prioriza estilo narrativo/periodístico) | Buena: recupera problema, impacto, causa, una acción correctiva y la solución final |
| Costo computacional | Inferencia seq2seq (generación token a token) | Solo *encoding* de oraciones (más liviano en CPU) |

### Análisis de resultados

El resumen abstractivo (`mT5_multilingual_XLSum`) alucinó en todos los intentos: inventó entidades, fechas y organizaciones que no están en el reporte, probablemente porque fue afinado sobre noticias periodísticas y no sobre texto técnico. En soporte técnico esto es riesgoso, porque un resumen con causas o soluciones inventadas puede llevar a decisiones equivocadas.

Por eso se probó un enfoque extractivo con el Transformer `paraphrase-multilingual-MiniLM-L12-v2`, usado solo para generar embeddings de las oraciones (no genera texto nuevo). Comparando estos embeddings contra 5 criterios semánticos (problema, impacto, causa, acciones, solución) mediante similitud coseno, se seleccionaron las oraciones más relevantes.

**Resultado real:** de las 12 oraciones del reporte (421 palabras) se seleccionaron **5**, produciendo un resumen de 170 palabras (**59.6% de reducción**) que sí conserva el problema, el impacto, la causa, una acción correctiva y la solución final.

**Conclusión:** al no generar texto nuevo, el resumen extractivo reduce mucho el riesgo de alucinación, aunque resulta menos fluido y compacto que uno abstractivo. La tabla de evaluación cualitativa queda pendiente de revisión humana antes de dar estas conclusiones por definitivas.


## 5. Problema 4 – Traducción técnica inglés → español

**Contexto:** buena parte de la documentación técnica (changelogs, mensajes de error, guías de despliegue) llega en inglés y debe ponerse a disposición del equipo de soporte hispanohablante de forma ágil.

**Modelo:** [`Helsinki-NLP/opus-mt-en-es`](https://huggingface.co/Helsinki-NLP/opus-mt-en-es), un modelo MarianMT entrenado específicamente para el par de idiomas inglés→español sobre el corpus OPUS. Resulta apropiado porque:
- Es un modelo **dedicado a un solo par de idiomas** (en vez de multilingüe genérico), lo que suele traducir con mayor precisión que modelos de propósito general.
- Es liviano y rápido en CPU, adecuado para traducir mensajes cortos de soporte en tiempo real.
- Al estar entrenado sobre un corpus amplio y variado (OPUS), maneja razonablemente bien tanto lenguaje cotidiano como parte de la terminología técnica de TI.

In [66]:
MODELO_TRADUCCION = "Helsinki-NLP/opus-mt-en-es"

traductor = pipeline(
    "translation",
    model=MODELO_TRADUCCION,
    device=DEVICE,
)


def traducir_texto(texto: str) -> str:
    """Traduce un texto técnico de inglés a español usando Helsinki-NLP/opus-mt-en-es."""
    resultado = traductor(texto)
    return resultado[0]["translation_text"]


# Prueba rápida de la función
traducir_texto("The server is currently unavailable.")

Device set to use cpu


'El servidor no está disponible actualmente.'

In [67]:
# 6 ejemplos con complejidad creciente y terminología técnica de soporte/desarrollo.
textos_traduccion = [
    "The server is currently unavailable.",
    "Please restart the service and check the endpoint status.",
    "The application failed to establish a connection with the PostgreSQL database.",
    "The reverse proxy returned a gateway timeout while communicating with the backend server.",
    "After deploying the latest release, users started experiencing intermittent authentication failures.",
    "The deployment pipeline rolled back automatically after the health check detected a database connection timeout on the new release.",
]

resultados_traduccion = []
for texto_original in textos_traduccion:
    traduccion = traducir_texto(texto_original)
    resultados_traduccion.append({"Texto original": texto_original, "Traducción generada": traduccion})
    print(f"EN: {texto_original}\nES: {traduccion}\n")

EN: The server is currently unavailable.
ES: El servidor no está disponible actualmente.

EN: Please restart the service and check the endpoint status.
ES: Por favor reinicie el servicio y compruebe el estado del endpoint.

EN: The application failed to establish a connection with the PostgreSQL database.
ES: La aplicación no pudo establecer una conexión con la base de datos PostgreSQL.

EN: The reverse proxy returned a gateway timeout while communicating with the backend server.
ES: El proxy inverso devolvió un tiempo de espera de puerta de enlace mientras se comunicaba con el servidor de backend.

EN: After deploying the latest release, users started experiencing intermittent authentication failures.
ES: Después de implementar la última versión, los usuarios comenzaron a experimentar fallos intermitentes de autenticación.

EN: The deployment pipeline rolled back automatically after the health check detected a database connection timeout on the new release.
ES: La tubería de despliegu

In [68]:
df_traduccion = pd.DataFrame(resultados_traduccion)
df_traduccion

,Texto original,Traducción generada
0,The server is currently unavailable.,El servidor no está disponible actualmente.
1,Please restart the service and check the endpoint status.,Por favor reinicie el servicio y compruebe el estado del endpoint.
2,The application failed to establish a connection with the PostgreSQL database.,La aplicación no pudo establecer una conexión con la base de datos PostgreSQL.
3,The reverse proxy returned a gateway timeout while communicating with the backend server.,El proxy inverso devolvió un tiempo de espera de puerta de enlace mientras se comunicaba con el servidor de backend.
4,"After deploying the latest release, users started experiencing intermittent authentication failures.","Después de implementar la última versión, los usuarios comenzaron a experimentar fallos intermitentes de autenticación."
5,The deployment pipeline rolled back automatically after the health check detected a database connection timeout on the new release.,La tubería de despliegue se rodó automáticamente después de que el chequeo de salud detectó un tiempo de conexión de la base de datos en la nueva versión.


### Análisis de resultados

**Conservación del significado y fluidez:** en los primeros 4 ejemplos (complejidad baja/media) la traducción es fluida y conserva completamente el significado: "endpoint", "PostgreSQL" y "backend" se mantienen tal cual (práctica común y correcta en terminología técnica), y "gateway timeout" se traduce como "tiempo de espera de puerta de enlace", una traducción literal pero comprensible y semánticamente correcta.

**Terminología técnica:**
- `endpoint`, `PostgreSQL`, `backend` → se conservan sin traducir, lo cual es apropiado porque son términos de uso estándar en español técnico.
- `deploying / release` → "implementar / última versión", traducción válida aunque en la jerga real de desarrollo en español se usa más frecuentemente "desplegar" y "release" (a veces sin traducir).
- `reverse proxy` → "proxy inverso", traducción correcta y estándar.

**Error identificado en el ejemplo más complejo (el de mayor longitud y densidad técnica):**

> EN: "The deployment pipeline **rolled back** automatically..."
> ES: "La tubería de despliegue **se rodó** automáticamente..."

Este es un **error de traducción real**: el modelo tradujo el verbo phrasal `rolled back` (revertir/deshacer un cambio) de forma **literal e incorrecta** como "se rodó" (que en español significa "giró/dio vueltas"), perdiendo por completo el sentido técnico de "rollback" (reversión automática de un despliegue). Además, tradujo `pipeline` como "tubería", una traducción literal que en español técnico normalmente se deja como "pipeline" o se traduce como "flujo/canalización de despliegue".

**Conclusión:** el modelo `Helsinki-NLP/opus-mt-en-es` es confiable para oraciones técnicas de complejidad baja a media, manteniendo apropiadamente términos técnicos en inglés cuando corresponde. Sin embargo, frente a **verbos compuestos idiomáticos** (`roll back`, y previsiblemente otros como `roll out`, `fall back`, `spin up`) tiende a traducir literalmente palabra por palabra, generando errores que cambian el significado técnico. Esto confirma la necesidad de revisión humana especializada para documentación técnica crítica, especialmente en oraciones largas con jerga de DevOps.